In [8]:
import pandas as pd
df = pd.read_csv("churn.csv")

In [9]:
df.shape

(7043, 21)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [11]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [12]:
print(df["Contract"].head())

0    Month-to-month
1          One year
2    Month-to-month
3          One year
4    Month-to-month
Name: Contract, dtype: object


In [13]:
print(df["customerID"].is_unique)   # True means every ID is unique

True


In [14]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [15]:
df.drop(columns=["customerID"], inplace=True)

In [16]:
df["TotalCharges"].isna().sum()

np.int64(11)

In [17]:
df["TotalCharges"].fillna(df["TotalCharges"].median(),inplace=True)

C:\Users\Saathwik Aithal\AppData\Local\Temp\ipykernel_1812\1795292956.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(),inplace=True)


In [18]:
df_cat = df.copy(deep=True)

In [19]:
ohe_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling",
    "PaymentMethod"
]

In [20]:
ord_cols = ["Contract"]

In [21]:
cat_features = ohe_cols + ord_cols

In [22]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])
df["Contract"] = ord_enc.fit_transform(df[["Contract"]])

In [23]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[ohe_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(ohe_cols),
    index=df.index
)

df = pd.concat([df.drop(columns=ohe_cols), encoded_df], axis=1)

In [24]:
df.shape

(7043, 44)

In [25]:
print(df["Churn"].value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [26]:
scale_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [28]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [29]:
from sklearn.model_selection import train_test_split

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
X.isna().sum()

SeniorCitizen                              0
tenure                                     0
Contract                                   0
MonthlyCharges                             0
TotalCharges                               0
gender_Female                              0
gender_Male                                0
Partner_No                                 0
Partner_Yes                                0
Dependents_No                              0
Dependents_Yes                             0
PhoneService_No                            0
PhoneService_Yes                           0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBack

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    r2_score
)
from sklearn.model_selection import cross_val_score

In [32]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred,pos_label=1))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("R2 Score :", r2_score(y_test, y_pred))

cv = cross_val_score(lr, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8055358410220014
Precision: 0.6572327044025157
Recall   : 0.5588235294117647
F1 Score : 0.6040462427745664
R2 Score : 0.002645379627476907
CV Scores: [0.80269695 0.81121363 0.7920511  0.81178977 0.80539773]
Mean CV Accuracy: 0.8046298349893541

Confusion Matrix
[[926 109]
 [165 209]]


In [33]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

cv = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8048261178140526
Precision: 0.6941176470588235
Recall   : 0.4732620320855615
F1 Score : 0.5627980922098569
ROC-AUC  : 0.8399480740912966
CV Scores: [0.80482612 0.80908446 0.77643719 0.80965909 0.79829545]
Mean CV Accuracy: 0.7996604619652881

Confusion Matrix
[[957  78]
 [197 177]]


In [34]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=1,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

cv = cross_val_score(xgb, X, y, cv=5, scoring="accuracy")

print("CV Scores:", cv)
print("Mean CV Accuracy:", cv.mean())

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8069552874378992
Precision: 0.6746575342465754
Recall   : 0.5267379679144385
F1 Score : 0.5915915915915916
ROC-AUC  : 0.846755276550673
CV Scores: [0.81476224 0.80269695 0.79134138 0.81036932 0.80184659]
Mean CV Accuracy: 0.8042032953738951

Confusion Matrix
[[940  95]
 [177 197]]


In [35]:
df_cat["Churn"] = df_cat["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [36]:
X_cat = df_cat.drop("Churn", axis=1)
y_cat = df_cat["Churn"]

In [37]:
from sklearn.model_selection import train_test_split

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

In [38]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=400,
    learning_rate=0.1,
    depth=4,
    random_seed=42,
    verbose=0
)

cat.fit(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features
)

CatBoostClassifier(depth=4, iterations=400, learning_rate=0.1, random_seed=42, verbose=0)

In [39]:
y_pred = cat.predict(X_test_cat)

print("Accuracy :", accuracy_score(y_test_cat, y_pred))
print("Precision:", precision_score(y_test_cat, y_pred))
print("Recall   :", recall_score(y_test_cat, y_pred))
print("F1 Score :", f1_score(y_test_cat, y_pred))
print("R2 Score :", r2_score(y_test_cat, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_cat, y_pred))

Accuracy : 0.8062455642299503
Precision: 0.6723549488054608
Recall   : 0.5267379679144385
F1 Score : 0.5907046476761619
R2 Score : 0.006285359993799977

Confusion Matrix
[[939  96]
 [177 197]]


In [40]:
from catboost import Pool, cv

train_pool = Pool(
    data=X_cat,
    label=y_cat,
    cat_features=cat_features
)

In [41]:
params = {
    "loss_function": "Logloss",
    "eval_metric": "Accuracy",
    "iterations": 400,
    "learning_rate": 0.1,
    "depth": 4,
    "random_seed": 42,
    "verbose": False
}

cv_results = cv(
    pool=train_pool,
    params=params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42
)

print(cv_results.tail())
print("Mean CV Accuracy:", cv_results["test-Accuracy-mean"].iloc[-1])

Training on fold [0/5]

bestTest = 0.8105039035
bestIteration = 65

Training on fold [1/5]

bestTest = 0.8197303052
bestIteration = 144

Training on fold [2/5]

bestTest = 0.8211497516
bestIteration = 207

Training on fold [3/5]

bestTest = 0.805535841
bestIteration = 61

Training on fold [4/5]

bestTest = 0.7967306326
bestIteration = 74

     iterations  test-Accuracy-mean  test-Accuracy-std  train-Accuracy-mean  \
395         395            0.797949           0.014737             0.841296   
396         396            0.797949           0.014822             0.841296   
397         397            0.797807           0.014717             0.841296   
398         398            0.798091           0.014857             0.841615   
399         399            0.797949           0.014966             0.841331   

     train-Accuracy-std  test-Logloss-mean  test-Logloss-std  \
395            0.002966           0.421282          0.016236   
396            0.003133           0.421329          0.01

In [42]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1, l2

In [43]:
model = Sequential([
    Dense(128, activation="relu",kernel_initializer="he_normal",kernel_regularizer=l2(0.001), input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu",kernel_initializer="he_normal",kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(32, activation="relu",kernel_initializer="he_normal",kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

C:\Users\Saathwik Aithal\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [44]:
model.compile(
    optimizer="SGD",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [45]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7247 - loss: 1.0003 - val_accuracy: 0.7720 - val_loss: 0.8991
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7495 - loss: 0.9511 - val_accuracy: 0.7702 - val_loss: 0.8907
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7586 - loss: 0.9234 - val_accuracy: 0.7782 - val_loss: 0.8942
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7628 - loss: 0.9128 - val_accuracy: 0.7773 - val_loss: 0.8903
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7704 - loss: 0.9034 - val_accuracy: 0.7791 - val_loss: 0.8863
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7706 - loss: 0.8988 - val_accuracy: 0.7791 - val_loss: 0.8799
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7808 - loss: 0.8893 - val_accuracy: 0.7720 - val_loss: 0.8745
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7826 - loss: 0.8801 - val_accu

In [46]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))


print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("Final Training Accuracy   :", history.history["accuracy"][-1])
print("Final Validation Accuracy :", history.history["val_accuracy"][-1])

print("Final Training Loss   :", history.history["loss"][-1])
print("Final Validation Loss :", history.history["val_loss"][-1])

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy : 0.7984386089425124
Precision: 0.631578947368421
Recall   : 0.5775401069518716
F1 Score : 0.6033519553072626
ROC-AUC  : 0.8393448552016328

Confusion Matrix
[[909 126]
 [158 216]]
Final Training Accuracy   : 0.8049700260162354
Final Validation Accuracy : 0.7914817929267883
Final Training Loss   : 0.6717959046363831
Final Validation Loss : 0.6872842907905579


In [55]:
import pandas as pd
import numpy as np

def predict_churn():

    inputs = [
        input("Gender(Male/ Female): "),
        int(input("Senior Citizen (0/1): ")),
        input("Partner (Yes/No): "),
        input("Dependents (Yes/No): "),
        int(input("Tenure (months): ")),
        input("Phone Service (Yes/No): "),
        input("Multiple Lines (Yes/No/No phone service): "),
        input("Internet Service (DSL/Fiber optic/No): "),
        input("Online Security (Yes/No/No internet service): "),
        input("Online Backup (Yes/No/No internet service): "),
        input("Device Protection (Yes/No/No internet service): "),
        input("Tech Support (Yes/No/No internet service): "),
        input("Streaming TV (Yes/No/No internet service): "),
        input("Streaming Movies (Yes/No/No internet service): "),
        input("Contract (Month-to-month/One year/Two year): "),
        input("Paperless Billing (Yes/No): "),
        input("Payment Method(Electronic check/ Mailed check/ Bank transfer (automatic)/ Credit card (automatic)): "),
        float(input("Monthly Charges(in K): ")),
        float(input("Total Charges(in K): "))
    ]

    columns = [
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "tenure",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "MonthlyCharges",
        "TotalCharges"
    ]

    new_data = pd.DataFrame([inputs], columns=columns)
    cat_data = new_data
    new_data = pd.get_dummies(new_data)

    new_data = new_data.reindex(columns=X.columns, fill_value=0)

    scale_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
    ]
    new_data_scaled = new_data.copy()

    new_data_scaled[scale_cols] = scaler.transform(new_data[scale_cols])

    lr_pred = lr.predict(new_data_scaled)[0]
    rf_pred = rf.predict(new_data_scaled)[0]
    xgb_pred = xgb.predict(new_data_scaled)[0]
    cb_pred = cat.predict(cat_data)[0]

    ann_prob = model.predict(new_data_scaled, verbose=0)[0][0]
    ann_pred = int(ann_prob >= 0.5)

    print("\n========== CHURN PREDICTIONS ==========")

    print("Logistic Regression :", "Churn" if lr_pred == 1 else "No Churn")
    print("Random Forest       :", "Churn" if rf_pred == 1 else "No Churn")
    print("XGBoost             :", "Churn" if xgb_pred == 1 else "No Churn")
    print("CatBoost             :", "Churn" if cb_pred == 1 else "No Churn")
    print("ANN                  :", "Churn" if ann_pred == 1 else "No Churn")

    print("\nANN Churn Probability:", round(ann_prob * 100, 2), "%")

In [56]:
predict_churn()

Gender(Male/ Female):  Male
Senior Citizen (0/1):  0
Partner (Yes/No):  Yes
Dependents (Yes/No):  Yes
Tenure (months):  12
Phone Service (Yes/No):  Yes
Multiple Lines (Yes/No/No phone service):  Yes
Internet Service (DSL/Fiber optic/No):  DSL
Online Security (Yes/No/No internet service):  Yes
Online Backup (Yes/No/No internet service):  Yes
Device Protection (Yes/No/No internet service):  Yes
Tech Support (Yes/No/No internet service):  Yes
Streaming TV (Yes/No/No internet service):  Yes
Streaming Movies (Yes/No/No internet service):  Yes
Contract (Month-to-month/One year/Two year):  One year
Paperless Billing (Yes/No):  Yes
Payment Method(Electronic check/ Mailed check/ Bank transfer (automatic)/ Credit card (automatic)):  Mailed check
Monthly Charges(in K):  70.54
Total Charges(in K):  844.43



========== CHURN PREDICTIONS ==========
Logistic Regression : No Churn
Random Forest       : No Churn
XGBoost             : No Churn
CatBoost             : No Churn
ANN                  : No Churn

ANN Churn Probability: 28.74 %
